
# Nebular backends: Cue, CloudyGrid, SSP-embedded, and BakedIn

Compare four nebular emission backends on identical star-forming spectra:

1. **Cue** — neural emulator (Li, Leja & Speagle 2023) over CLOUDY parameter space
2. **CloudyGrid** — traditional CLOUDY photoionization grid (if available)
3. **SSP-embedded** — nebular lines baked into SSP templates (BakedIn / Byler)
4. **BakedIn** — wNE SSP with emission in continuum (compare line shapes with Cue)

Shows [OIII] 5007 + H-beta and Hα regions. BakedIn and Cue have
different ionization flexibility — BakedIn pulls lines from SSP metallicity
grid, Cue samples ionization parameter (log U) independently.

References:

- Li, Leja & Speagle 2023, ApJ, 956, 23 (Cue neural emulator)
- Byler et al. 2017, ApJ, 840, 44 (BakedIn SSP-embedded lines)


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

ssp_bare = tengri.load_ssp("fsps_prsc_miles_chabrier")
ssp_wne = tengri.load_ssp()  # wNE: BakedIn needs wNE SSP

# Common SFH/dust/redshift config
common_config = {
    "sfh": {
        "type": "dpl",
        "all_params": tengri.FIXED,
        "alpha": 1.0,
        "beta": 2.5,
        "tau_gyr": 0.5,
        "log_total_mass": 10.0,
    },
    "dust": {"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0},
    "redshift": tengri.Fixed(0.0),
}

# Build Cue (bare-stellar SSP)
model_cue = tengri.SEDModel.build(
    ssp_bare,
    neb={"type": "cue", "all_params": tengri.FIXED, "neb_logU": tengri.Fixed(-3.0)},
    **common_config,
)

# Build BakedIn (wNE SSP with nebular lines embedded)
model_baked = tengri.SEDModel.build(
    ssp_wne,
    neb={"type": "ssp", "all_params": tengri.FIXED},
    **common_config,
)

params_cue = dict(model_cue.spec.sample(jax.random.PRNGKey(0)))
params_baked = dict(model_baked.spec.sample(jax.random.PRNGKey(0)))

out_cue = model_cue.predict(params_cue)
out_baked = model_baked.predict(params_baked)

# The two backends use different SSP grids (bare-stellar vs wNE), so each
# SED carries its own wavelength array — never share a single mask between them.
wave = np.asarray(model_cue.wavelengths)
sed_cue = np.asarray(out_cue.rest_sed())
wave_baked = np.asarray(model_baked.wavelengths)
sed_baked = np.asarray(out_baked.rest_sed())

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

regions = [
    (axes[0], 4700, 5100, r"[O III] + H$\beta$ region", 4861, 5007),
    (axes[1], 6400, 6750, r"H$\alpha$ region", None, 6564.61),
]

for ax, wmin, wmax, _title, lam_hbeta, lam_main in regions:
    mask = (wave > wmin) & (wave < wmax)
    mask_baked = (wave_baked > wmin) & (wave_baked < wmax)

    # Cue (bare-stellar, neural emulator)
    ax.plot(
        np.array(wave[mask]),
        np.array(sed_cue[mask]),
        "k-",
        lw=1.8,
        label="Cue (logU-flexible)",
    )

    # BakedIn (wNE SSP, embedded lines) — on its own grid.
    # Peak-normalize BakedIn to Cue for shape comparison.
    cue_peak = np.nanmax(sed_cue[mask])
    baked_peak = np.nanmax(sed_baked[mask_baked])
    if baked_peak > 0:
        sed_baked_norm = sed_baked[mask_baked] * (cue_peak / baked_peak)
        ax.plot(
            np.array(wave_baked[mask_baked]),
            sed_baked_norm,
            "C3--",
            lw=1.5,
            alpha=0.7,
            label="BakedIn (SSP-embedded)",
        )

    if lam_hbeta is not None:
        ax.axvline(lam_hbeta, ls=":", color="C1", lw=0.8, alpha=0.6)
        ax.text(lam_hbeta + 5, ax.get_ylim()[1] * 0.9, r"H$\beta$", fontsize=9, color="C1")
    ax.axvline(lam_main, ls=":", color="C2", lw=0.8, alpha=0.6)
    label_main = r"[O III]" if lam_main == 5007 else r"H$\alpha$"
    ax.text(lam_main + 5, ax.get_ylim()[1] * 0.8, label_main, fontsize=9, color="C2")

    ax.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]", fontsize=11)
    ax.set_ylabel(r"$L_\nu$ [erg/s/Hz]", fontsize=11)
    ax.legend(frameon=False, fontsize=10, loc="upper right")

fig.tight_layout()
plt.savefig("plot_nebular_backends.png", dpi=150, bbox_inches="tight")